In [62]:
"""
SEC Filing Scraper
@Author: Nicholas Jung
"""

import requests
import pandas as pd

In [63]:
headers = {'User-Agent':
          "nicholasjung0503@gmail.com"}

target_tickers = ['LULU', 'UAA']

In [64]:
print("getting the CIK master list..")
tickers_url = "https://www.sec.gov/files/company_tickers.json"
ciks_dict = requests.get(
    tickers_url,
    headers = headers
)
df_ciks = pd.DataFrame.from_dict(ciks_dict.json(), orient= "index")
df_ciks['cik_str']= df_ciks['cik_str'].astype(str).str.zfill(10)

getting the CIK master list..


### Pulling the XBRL tags

In [92]:
metrics_to_pull = {
    'Net_Revenue': ['Revenues', 'RevenueFromContractWithCustomerExcludingAssessedTax', 'SalesRevenueNet'],
    'Gross_Profit': ['GrossProfit'],
    'Operating_Income': ['OperatingIncomeLoss'],
    'Net_Income': ['NetIncomeLoss']
}



### Little bit of logic
- since APIs are fragile, have to implement at try/except block that acts as a safety net. Otherwise, if a file is corrupted then the entire thing crashes, losing all your progress at the same time.


In [116]:
allCompanyData = []
for ticker in target_tickers:
    print(f"Processing {ticker}..")
    try:
        cik = df_ciks[df_ciks["ticker"] == ticker]['cik_str'].iloc[0]
        facts_url = f'https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json'
        companyFacts = requests.get( facts_url,
                                     headers = headers
                                   )
        ## saves raw dictionary
        usgaapCompany = companyFacts.json()['facts']['us-gaap']

        for metric_name, xbrl_tags in metrics_to_pull.items():
            print(f"Hunting for: {metric_name}...")

            valid_tag = None

            for tag in xbrl_tags:
                if tag in usgaapCompany:
                    valid_tag = tag
                    break
            if valid_tag:
                print(f"Success : we found {valid_tag}")
                raw_data = usgaapCompany[valid_tag]['units']['USD']

                df_metric = pd.DataFrame(raw_data)
                df_quarterly = df_metric[df_metric['form'] == "10-Q"].copy() ##
                
                if 'segment' in df_quarterly.columns:
                    df_quarterly = df_quarterly[df_quarterly['segment'].isna()]
                    
                df_quarterly = df_quarterly.reset_index(drop = True)
                ## slicing two columns, makign sure we rename
                df_quarterly = df_quarterly[['end', 'val']].rename(columns = {'end': 'Date', 'val': 'Value'})
                df_quarterly['Ticker'] = ticker
                df_quarterly['Metric'] = metric_name

                                                                       
                allCompanyData.append(df_quarterly)

            else:
                print(f" Warning: Could not find SEC tag for {metric_name}")
        
    except Exception as e:
        print(f"Error processing {ticker}: {e}")

print(allCompanyData)

Processing LULU..
Hunting for: Net_Revenue...
Success : we found RevenueFromContractWithCustomerExcludingAssessedTax
Hunting for: Gross_Profit...
Success : we found GrossProfit
Hunting for: Operating_Income...
Success : we found OperatingIncomeLoss
Hunting for: Net_Income...
Success : we found NetIncomeLoss
Processing UAA..
Hunting for: Net_Revenue...
Success : we found Revenues
Hunting for: Gross_Profit...
Success : we found GrossProfit
Hunting for: Operating_Income...
Success : we found OperatingIncomeLoss
Hunting for: Net_Income...
Success : we found NetIncomeLoss
[          Date       Value Ticker       Metric
0   2018-10-28  2120861000   LULU  Net_Revenue
1   2018-10-28   747655000   LULU  Net_Revenue
2   2019-05-05   782315000   LULU  Net_Revenue
3   2019-08-04  1665667000   LULU  Net_Revenue
4   2019-08-04   883352000   LULU  Net_Revenue
..         ...         ...    ...          ...
59  2025-05-04  2370660000   LULU  Net_Revenue
60  2025-08-03  4895879000   LULU  Net_Revenue
61

### Submitting all into one, clean dataset.

In [120]:
pd.concat(allCompanyData)

,Date,Value,Ticker,Metric
0,2018-10-28,2120861000,LULU,Net_Revenue
1,2018-10-28,747655000,LULU,Net_Revenue
2,2019-05-05,782315000,LULU,Net_Revenue
3,2019-08-04,1665667000,LULU,Net_Revenue
4,2019-08-04,883352000,LULU,Net_Revenue
...,...,...,...,...
153,2025-06-30,-2612000,UAA,Net_Income
154,2025-09-30,-21426000,UAA,Net_Income
155,2025-09-30,-18814000,UAA,Net_Income
156,2025-12-31,-452253000,UAA,Net_Income


In [117]:
df_final = pd.concat(allCompanyData)
df_final = df_final.reset_index(drop=True)
print(df_final.columns)

Index(['Date', 'Value', 'Ticker', 'Metric'], dtype='object')


In [119]:
df_final['Metric']

0       Net_Revenue
1       Net_Revenue
2       Net_Revenue
3       Net_Revenue
4       Net_Revenue
           ...     
1085     Net_Income
1086     Net_Income
1087     Net_Income
1088     Net_Income
1089     Net_Income
Name: Metric, Length: 1090, dtype: object